# Le second témoin : la productivité globale des facteurs · *The second witness: total factor productivity*

Notebook compagnon du chapitre **13. IA, automatisation et productivité : un nouveau moteur de croissance ?** — [lire l'article](https://nmlab.io/ressources/ia-automatisation-productivite).
Companion notebook to chapter **13. AI, Automation and Productivity: A New Engine of Growth?** — [read the article](https://nmlab.io/en/ressources/ai-automation-and-productivity).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# données FRED chargées dans build_figure


from matplotlib.figure import Figure
import matplotlib.pyplot as plt
import pandas as pd

C, W = nm.COLORS, nm.WIDTH_PX


def wrap(ax, x: float, y: float, text: str, *, size: float = 19, color: str | None = None,
         weight: int = 500, ha: str = "left", va: str = "top", width: int = 42,
         lh: float = 1.5) -> int:
    """Écrit un texte replié à ``width`` caractères (coordonnées pixels)."""
    import textwrap
    lines: list[str] = []
    for para in text.split("\n"):
        lines += textwrap.wrap(para, width) or [""]
    ax.text(x, y, "\n".join(lines), fontsize=size, color=color or C["muted"],
            fontweight=weight, ha=ha, va=va, linespacing=lh, zorder=5)
    return len(lines)


def start(height: int = 1010) -> Figure:
    """Figure NMLab au format du site : 1747 px de large, fond sombre."""
    fig = nm.figure(height_px=height)
    fig.patch.set_facecolor(C["bg"])
    return fig


def dec(v: float, lang: str, n: int = 1, sign: bool = False) -> str:
    """Formate un nombre à la française (virgule, moins typographique) ou à l'anglaise."""
    s = f"{v:+.{n}f}" if sign else f"{v:.{n}f}"
    return s.replace("-", "−").replace(".", ",") if lang == "fr" else s


LABELS = {
    "fr": dict(
        title='Le second témoin : la productivité globale des facteurs',
        sub='Croissance annuelle de la PTF américaine, et sa moyenne par période',
        l1='moyenne 2006-2019',
        l2='moyenne depuis 2020',
        note="BLS (MFPNFBS) via FRED. La PTF — le « progrès pur » du chapitre 12 — a doublé de rythme,\nce qui corrobore l'accélération sans rien dire de sa cause.",
    ),
    "en": dict(
        title='The second witness: total factor productivity',
        sub='Annual growth of US total factor productivity, and its period averages',
        l1='2006-2019 average',
        l2='average since 2020',
        note="BLS (MFPNFBS) via FRED. TFP — chapter 12's « pure progress » — has doubled its pace,\nwhich corroborates the acceleration without saying anything about its cause.",
    ),
}


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab (libellés selon ``lang``)."""
    t = LABELS[lang]
    s = nm.load_fred("MFPNFBS").pct_change().dropna() * 100
    s = s[s.index >= "2000-01-01"]
    fig = start(1010); ax = nm.axes(fig, left=0.075, bottom=0.19)
    nm.header(fig, t["title"], t["sub"])
    cols = [C["blue"] if d.year <= 2019 else C["amber"] for d in s.index]
    ax.bar(s.index.year, s.values, color=cols, width=0.62, zorder=3)
    a = s[(s.index.year >= 2006) & (s.index.year <= 2019)].mean()
    b = s[s.index.year >= 2020].mean()
    ax.plot([2005.6, 2019.4], [a, a], color=C["blue"], lw=3, ls="--", zorder=4)
    ax.plot([2019.6, s.index.year.max() + 0.4], [b, b], color=C["amber"], lw=3, ls="--", zorder=4)
    dec = "," if lang == "fr" else "."
    ax.text(2012.5, a + 0.42, f"{t['l1']} : {a:.2f}".replace(".", dec) + " %",
            color=C["blue"], fontsize=19.5, fontweight=700, ha="center")
    ax.text(2022.8, b + 0.55, f"{t['l2']} : {b:.2f}".replace(".", dec) + " %",
            color=C["amber"], fontsize=19.5, fontweight=700, ha="center")
    ax.axhline(0, color=C["edge"], lw=1.6)
    ax.grid(axis="y", color=C["grid"], lw=1.2); ax.set_axisbelow(True)
    ax.tick_params(labelsize=18, colors=C["muted"], length=0)
    ax.set_ylim(min(s.min() - 0.8, -2.2), max(s.max() + 1.1, 3.2))
    for sp in ("top", "right", "bottom", "left"): ax.spines[sp].set_visible(False)
    nm.footer(fig, t["note"])
    return fig


build_figure(LANG)